# VISCERA — FINAL pipeline: **GastroNet-DINOv2 + VLM-Concept Teaching (Stage-1, GRL nuisance-suppression) + CG-AMIL @448 + SEMI + color-aug OOD**  (Run-All → ship)

Winning recipe (evidence: GastroNet-DINOv2 frozen-LP cross-center 0.93 ≫ generic DINOv3 0.835; exps/2 dinov2 hidden 0.854 > exps/3 dinov3 0.756).
**Method = 3 pillars:** (1) *Semi-supervised* Mean-Teacher+PU on the 144k pool; (2) *VLM-Concept Teaching* — Stage-1 distills 35 clinical concepts, diagnostic→trunk / nuisance→GRL (the OOD layer that ignores acquisition/scope shortcuts); (3) *OOD generalization* levers: color/FDA aug (`--aug domain`), MixStyle, WiSE-FT.
**Gate first** (cell 10, `RUN_GATE=True`): confirm the OOD bundle beats baseline on the held-out-center proxy before the single ship.

In [ ]:
import torch; print(torch.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install timm==1.0.27 scikit-learn

2.11.0+cu128
name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


In [ ]:
# MUST match the repo you `git push` to, or Colab silently trains on STALE code. `origin` is VISCERA.git,
# so this is VISCERA.git (commit 2cba078 already made this call; it got reverted by a notebook round-trip).
# If you switch to RARE2026.git, change `origin` too so push and clone can never drift apart again.
REPO_URL = 'https://github.com/HuynhDoTanThanh/VISCERA.git'   # <-- must equal `git remote -v` origin
%cd /content
!rm -rf rare && git clone $REPO_URL rare
%cd /content/rare

/content
Cloning into 'rare'...
remote: Enumerating objects: 437, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 437 (delta 70), reused 65 (delta 32), pack-reused 332 (from 1)
Receiving objects: 100% (437/437), 709.15 KiB | 19.70 MiB/s, done.
Resolving deltas: 100% (247/247), done.
/content/rare


In [ ]:
# ---- HF token for the gated DINOv3 download (needed ONCE to fetch dinov3.pth; then cached to Drive) ----
# SECURE: put the token in Colab's Secrets panel (left sidebar, key icon) as name HF_TOKEN — do NOT paste it here
# (this notebook is pushed to GitHub; a committed token leaks publicly and GitHub push-protection will block it).
import os
try:
    from google.colab import userdata
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))   # read from Colab Secrets, never hard-coded
except Exception:
    pass   # not on Colab, or Secret not set -> fine if dinov3.pth is already cached on Drive
print('HF_TOKEN set' if os.environ.get('HF_TOKEN') else 'HF_TOKEN not set (ok if dinov3.pth already on Drive)')

HF_TOKEN not set (ok if dinov3.pth already on Drive)


In [ ]:
# ---- config + data (backbone weights + numbered data zips) ----
from google.colab import drive; drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/RARE_LG'   # <-- your Drive folder
BACKBONE  = 'dinov2'                            # 'dinov3' (ViT-B/16, stronger dense features + CG-AMIL) | 'dinov2' (exp1 path)
FT_IMG    = 336                                 # THE resolution. Stage-1 AND Stage-2 MUST use it (4-experiment
                                                # LB signal: @336 > @448). Stage-1 used to inherit featurize.IMG=448
                                                # silently -> a 448 concept encoder under a 336 ship. Never split these.
import glob, os, zipfile, shutil
# backbone weights: dinov2.pth (SSL teacher) or dinov3.pth (timm state_dict). Cache on Drive; dinov3 downloaded once.
if BACKBONE == 'dinov2':
    assert os.path.exists(f'{DRIVE_DIR}/dinov2.pth'), f'upload dinov2.pth to {DRIVE_DIR}'
    shutil.copy(f'{DRIVE_DIR}/dinov2.pth', 'dinov2.pth'); print('dinov2.pth OK')
else:
    if os.path.exists(f'{DRIVE_DIR}/dinov3.pth'):
        shutil.copy(f'{DRIVE_DIR}/dinov3.pth', 'dinov3.pth'); print('REUSED dinov3.pth from Drive')
    else:
        # gated download -> set os.environ['HF_TOKEN']='hf_...' in a cell ABOVE (do NOT commit the token to git)
        import timm, torch
        m = timm.create_model('vit_base_patch16_dinov3', pretrained=True, img_size=448, num_classes=0)
        torch.save(m.state_dict(), 'dinov3.pth'); shutil.copy('dinov3.pth', f'{DRIVE_DIR}/dinov3.pth')
        print('downloaded + cached dinov3.pth')
os.makedirs('out', exist_ok=True)
def extract_chunk(zpath):
    with zipfile.ZipFile(zpath) as zf:
        tops = {p.split('/')[0] for p in zf.namelist() if p.strip('/')}
        target = f"out/{os.path.splitext(os.path.basename(zpath))[0]}" if ('images' in tops or 'labels' in tops) else ('.' if tops=={'out'} else 'out')
        os.makedirs(target, exist_ok=True); zf.extractall(target)
for z in [p for p in sorted(glob.glob(f'{DRIVE_DIR}/*.zip')) if 'dataset' not in os.path.basename(p).lower()]:
    extract_chunk(z); print('extracted', os.path.basename(z))
print('out dirs:', len(glob.glob('out/*/labels')), '| train labels:', len(glob.glob('out/train/labels/*.json')))

Mounted at /content/drive
dinov2.pth OK
extracted 0.zip
extracted 1.zip
extracted 10.zip
extracted 11.zip
extracted 12.zip
extracted 13.zip
extracted 14.zip
extracted 15.zip
extracted 16.zip
extracted 17.zip
extracted 18.zip
extracted 19.zip
extracted 2.zip
extracted 20.zip
extracted 21.zip
extracted 22.zip
extracted 23.zip
extracted 24.zip
extracted 26.zip
extracted 3.zip
extracted 33.zip
extracted 34.zip
extracted 35.zip
extracted 36.zip
extracted 37.zip
extracted 38.zip
extracted 39.zip
extracted 4.zip
extracted 40.zip
extracted train.zip
extracted val.zip
out dirs: 31 | train labels: 2476


In [ ]:
# ---- labeled CSVs from the JSON labels (img = out/<split>/images/<name>.jpg) ----
import json, glob, os, csv
def build_csv(split):
    rows=[]
    for j in glob.glob(f'out/{split}/labels/*.json'):
        d=json.load(open(j))
        if int(d.get('label',-1))<0: continue
        name=d.get('name', os.path.splitext(os.path.basename(j))[0]); img=f'out/{split}/images/{name}.jpg'
        if os.path.exists(img): rows.append({'path':img,'center':d.get('center',''),'class':'','label':int(d['label'])})
    with open(f'{split}_colab.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['path','center','class','label']); w.writeheader(); w.writerows(rows)
    print(f'{split}_colab.csv', len(rows),'pos=',sum(r['label'] for r in rows),'centers=',sorted({r['center'] for r in rows}))
build_csv('train')
try: build_csv('val')
except Exception as e: print('val skip', e)

train_colab.csv 2476 pos= 127 centers= ['center_1', 'center_2']
val_colab.csv 619 pos= 31 centers= ['center_1', 'center_2']


In [ ]:
# ---- 35-concept target matrix for Stage-1. VALIDATED, not blindly reused. ----
# The matrix stores absolute-ish image paths. A copy built on a laptop stores the LABELED rows as
# 'dataset/train/<cls>/<id>.png', but Colab only unzips 'out/' -> those 2,476 rows (incl. ALL 127 positives)
# would not resolve. pretrain_concept used to swallow the read error and feed a BLACK image with the real
# concept targets, for 30 epochs. It now hard-fails; this cell rebuilds instead of letting that happen.
import os, shutil, numpy as np
os.makedirs('phase3/cache', exist_ok=True)
LOCAL, DRIVE_CT = 'phase3/cache/concept_targets.npz', f'{DRIVE_DIR}/concept_targets.npz'
def paths_resolve(npz, n=400):
    try: z = np.load(npz, allow_pickle=True)
    except Exception: return False
    p = z['paths'].astype(str); cen = z['center'].astype(str)
    idx = np.concatenate([np.where(cen != '')[0][:200], np.arange(0, len(p), max(1, len(p)//n))])
    miss = [q for q in p[np.unique(idx)] if not os.path.exists(q)]
    if miss: print(f'  {len(miss)} sampled paths MISSING here (e.g. {miss[:2]})')
    return not miss
if os.path.exists(LOCAL) and paths_resolve(LOCAL):
    print('concept_targets.npz present and paths resolve.')
elif os.path.exists(DRIVE_CT) and (shutil.copy(DRIVE_CT, LOCAL), paths_resolve(LOCAL))[1]:
    print('REUSED concept_targets.npz from Drive (paths validated)')
else:
    print('rebuilding concept_targets.npz on THIS machine (cached copy unusable / paths do not resolve) ...')
    !python -m phase3.build_concept_targets --out {LOCAL}
    assert paths_resolve(LOCAL), 'rebuilt matrix still has unresolvable paths — check that out/ is fully extracted'
    shutil.copy(LOCAL, DRIVE_CT); print('built + cached concept_targets.npz')


REUSED concept_targets.npz from Drive (paths validated)


## Stage-1 — concept-supervised pretraining (full-35 losses: certain / uncertain / smoothing)

LIGHT + L2-SP anchor → *shape*, don't overwrite SSL. Upgrades vs the old masked-BCE:
- **`--discrim full15`** — the full discriminative core (9 → 15; adds `color_heterogeneity, whitish_focal_area, vascular_irregularity, dilated_vessels, focal_abnormal_vessels, border_sharpness`). This is the real substance of "more labels".
- **certain** = class-balanced soft-BCE (`--pos_weight_cap 10` rebalances rare-positive concepts, e.g. `depression_ulceration` p=.055 → 10× — the term most likely to move PPV@90R).
- **uncertain** (`--unc_w 0.1`) = prior-pull on `not_assessable` cells instead of hard-masking (never invents a label).
- **smoothing** (`--smooth_eps 0.05`) = target shrinks toward the per-concept prior so the head can't get more confident than the noisy VLM warrants (anti center-memorization).

Every knob **defaults to the old masked-BCE** → clean superset, ablatable via the flags. Note: **7 of the 35 concepts are dead constants** (value ≡ 0: modality/distance/view/landmark/interpretable_fraction/dominant_color/lesion_size) and are auto-dropped; context/acquisition concepts go to a **detached AUX head** so "all 35" never re-injects center style into the trunk.

In [ ]:
RUN_GATES = False   # True = also build the leak-free gating encoder (one extra 30-epoch Stage-1)
# Stage-1: concept-supervised pretraining -> concept_encoder.pt (per-backbone+RESOLUTION cache; 30 epochs).
# CACHE KEY NOW INCLUDES FT_IMG. Before this fix the key was concept_encoder_{BACKBONE}.pt with no resolution, so a
# 448-era encoder was silently reused under a @336 ship — an uncontrolled second variable on top of --aug domain,
# and @448 is precisely what the leaderboard proved regressive (docs section 8). Stage-1 also had no --img flag at
# all and inherited featurize.IMG (=448 since commit f4271e2); both are fixed now.
import os, shutil
CE_DRIVE = f'{DRIVE_DIR}/concept_encoder_{BACKBONE}_{FT_IMG}.pt'
LEGACY   = f'{DRIVE_DIR}/concept_encoder_{BACKBONE}.pt'          # unversioned, resolution unknown
if os.path.exists(CE_DRIVE):
    shutil.copy(CE_DRIVE, 'concept_encoder.pt'); print(f'REUSED concept_encoder_{BACKBONE}_{FT_IMG}.pt -> Stage-1 SKIPPED')
else:
    if os.path.exists(LEGACY):
        print(f'NOTE: found legacy {LEGACY} with NO resolution in its name. Not reusing it — its provenance is '
              f'unverifiable and a 448 encoder under a {FT_IMG} ship would confound the experiment. Rebuilding @{FT_IMG}.')
    print(f'running Stage-1 ({BACKBONE} @{FT_IMG}, 30 epochs) ...')
    !python -m phase3.pretrain_concept --targets phase3/cache/concept_targets.npz --backbone {BACKBONE} \
        --img {FT_IMG} \
        --unfreeze 3 --epochs 30 --bs 128 --lr 1e-4 --grl 1.0 --l2sp 1.0 --workers 8 \
        --discrim full15 --context_route detach --certain_floor 0.7 --smooth_eps 0.05 --unc_w 0.1 --pos_weight_cap 10 \
        --out concept_encoder.pt
    shutil.copy('concept_encoder.pt', CE_DRIVE); print(f'saved concept_encoder_{BACKBONE}_{FT_IMG}.pt')
# ---- gating encoder: Stage-1 with the 2,476 LABELED frames REMOVED -------------------------------
# concept_targets.npz contains those labeled frames (center_1 1823 / center_2 653) — the exact images a LOCO
# leg evaluates on — supervised with concepts that are 0.87-0.91 AUROC proxies for the neo label. So a
# `--init concept_encoder.pt` LOCO run is optimistic even with --loco-no-semi. One --holdout labeled encoder
# (drops 1.45% of the corpus) is honest for BOTH legs. Ship uses the full encoder; GATES use this one.
CE_GATE = f'{DRIVE_DIR}/concept_encoder_{BACKBONE}_{FT_IMG}_gate.pt'
if RUN_GATES:
    if os.path.exists(CE_GATE):
        shutil.copy(CE_GATE, 'concept_encoder_gate.pt'); print('REUSED leak-free gating encoder')
    else:
        print(f'running Stage-1 GATING encoder ({BACKBONE} @{FT_IMG}, --holdout labeled) ...')
        !python -m phase3.pretrain_concept --targets phase3/cache/concept_targets.npz --backbone {BACKBONE} \
            --img {FT_IMG} --holdout labeled \
            --unfreeze 3 --epochs 30 --bs 128 --lr 1e-4 --grl 1.0 --l2sp 1.0 --workers 8 \
            --discrim full15 --context_route detach --certain_floor 0.7 --smooth_eps 0.05 --unc_w 0.1 --pos_weight_cap 10 \
            --out concept_encoder_gate.pt
        shutil.copy('concept_encoder_gate.pt', CE_GATE); print('saved leak-free gating encoder')

# provenance check — finetune.py also warns, but fail early here rather than after a 3-seed ship
import torch
_cfg = (torch.load('concept_encoder.pt', map_location='cpu', weights_only=False).get('cfg') or {})
print(f"concept encoder img={_cfg.get('img', 'UNKNOWN (pre---img build)')} | ship img={FT_IMG}")
assert _cfg.get('img', FT_IMG) == FT_IMG, f"Stage-1 @{_cfg.get('img')} != ship @{FT_IMG} — rebuild Stage-1"


REUSED concept_encoder_dinov2_336.pt -> Stage-1 SKIPPED
concept encoder img=336 | ship img=336


## Stage-2 — FINAL ship (exp6): **GastroNet-DINOv2 @336 + `--aug domain`** (color/acquisition randomization)
exp6 = the best board recipe (**exps/2 @336 = 0.0177**) with ONE change: `aug mild → domain`. Grounded in a full code audit (2026-07-23):
- **@336, not @448** — 4-experiment AUROC signal: @336 (0.845, 0.854) > @448 (0.797, 0.829); @448 adds center-specific hi-freq texture (§8). exp5's @448 regression proved this.
- **`--aug domain`** = AcquisitionAug (white-balance + HSV + FDA + gamma) attacks the per-center **color axis** = the root 0.996 center-separability. Critically, it also flips the **288k semi-consistency strong view** to AcquisitionAug — so the 300k pool finally teaches invariance to a NEW center's color nuisance (the winner's color-aug lever, never tried at @336).
- **Honest gate first** (cell after the ship): `--loco-no-semi` A/B of mild vs aug-domain @336 — leak-free (the semi pool has no center label and otherwise leaks the held-out center → UDA to the test center → optimistic LOCO; frozen-LP `loco_probe.py` is the compass that actually predicted the leaderboard).

In [ ]:
# ---- DE-RISK GATE — NOW LEAK-FREE (--loco-no-semi + --holdout labeled encoder) and @FT_IMG, not @448.
# ---- Also note: --cg-head was NEVER TRAINED before 2026-08 (AttnPool was missing from the optimizer),
# ---- so any previous BUNDLE verdict measured random attention. Re-run if you care about that arm.
# ---- DE-RISK GATE (run ONCE before the single submission) — does the OOD bundle beat baseline on the NEW-center proxy? ----
# BASE   = dinov2@448 + concept-init + SEMI + mild aug + mean-pool  (the safe exps/2-style recipe)
# BUNDLE = BASE + --aug domain (color/FDA OOD) + --mixstyle + --cg-head (attention-MIL)   (the winning method)
# Trains both with each center held out; paired AUROC/AUPRC gate. Ships NOTHING. Set RUN_GATE=True to spend ~4 finetunes (~1h).
RUN_GATE = False
if RUN_GATE:
    import os, shutil, numpy as np, phase3.evaluate as ev
    os.makedirs('phase3/cache', exist_ok=True)
    if not os.path.exists('phase3/cache/unl_manifest.npz'):
        if os.path.exists(f'{DRIVE_DIR}/unl_manifest.npz'):
            shutil.copy(f'{DRIVE_DIR}/unl_manifest.npz', 'phase3/cache/unl_manifest.npz')
        else:
            !python -m phase3.mine_hardneg --manifest-only
            shutil.copy('phase3/cache/unl_manifest.npz', f'{DRIVE_DIR}/unl_manifest.npz')
    SEMI = ('--semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 '
            '--semi-n 300000 --semi-bs 192 --semi-steps 10')
    BASE   = f'--backbone dinov2 --img {FT_IMG} --init concept_encoder_gate.pt --unfreeze 6 --wise-ft 0.7 --epochs 12 --bs 96 --loss bce+rank+pauc --warmup 2 --loco-no-semi {SEMI}'
    BUNDLE = BASE + ' --aug domain --mixstyle --cg-head --attn-entropy 0.1 --attn-floor 0.5'
    for hold in ['center_2', 'center_1']:
        for name, flags in [('base', BASE), ('bundle', BUNDLE)]:
            print(f'--- holdout {hold} : {name} ---')
            !python -m phase3.finetune --train-csv train_colab.csv --seed 0 --holdout {hold} {flags} --out loco_{name}_{hold}.pt
    def leg(h, n):
        d = np.load(f'loco_{n}_{h}_loco.npz', allow_pickle=True); return d['y'], d['c'], d['s']
    fails = []
    for hold in ['center_2', 'center_1']:
        y, c, sb = leg(hold, 'bundle'); _, _, sa = leg(hold, 'base')
        print(f'\nholdout {hold}  (BUNDLE vs BASE):')
        for m in ('auroc', 'auprc'):
            g = ev.gate(y, sb, sa, center=c, metric=m, B=2000)
            if g['verdict'] == 'FAIL': fails.append((hold, m))
            print(f"  {m}: Δ={g['delta']:+.4f} CI[{g['lo']:+.4f},{g['hi']:+.4f}] -> {g['verdict']}")
    print('\nDECISION:', f'BUNDLE regresses at {fails} -> ship BASE (drop OOD levers)' if fails
          else 'BUNDLE holds/improves on BOTH new-center legs -> ship the winning bundle (cell 11 is correct)')
else:
    print('Gate SKIPPED. Recommend RUN_GATE=True once: it tells you if --aug domain/--mixstyle/--cg-head actually help the unseen center BEFORE you spend the submission.')


Gate SKIPPED. Recommend RUN_GATE=True once: it tells you if --aug domain/--mixstyle/--cg-head actually help the unseen center BEFORE you spend the submission.


In [ ]:
# ---- SUPERSEDED by MAX-A/B/C (cells 13-15). This exp6 cell also writes ship_seed*.pt, so a
# ---- Run-All would clobber the MAX ensemble. Set RUN_LEGACY=True only to reproduce exp6.
RUN_LEGACY = False
if not RUN_LEGACY:
    print('SKIPPED: exp6 cell is superseded by MAX-A/B/C. Set RUN_LEGACY=True to run it.')
else:
    # ==== FINAL SHIP (exp6): exps/2 recipe @336 + --aug domain (color/acquisition randomization) ====
    # ONE variable changed vs the best board score (exps/2 @336 = 0.0177): aug mild -> domain. Grounded in the code audit:
    #   * @336 (NOT @448): 4-experiment AUROC signal @336 > @448 (section 8) — @448 adds center-specific hi-freq texture.
    #     Stage-1 is now ALSO pinned to FT_IMG (cell 8) — it used to run @448 regardless, confounding this experiment.
    #   * --aug domain: AcquisitionAug (white-balance+HSV+FDA+gamma) attacks the per-center COLOR axis = the root
    #     0.996 center-separability. It ALSO switches the semi-consistency STRONG view to AcquisitionAug, so the
    #     unlabeled pool finally teaches invariance to the exact nuisance a NEW center introduces.
    #   * everything else identical to exps/2: mean-pool, concept-init, unfreeze 6, WiSE-FT 0.7, 12ep, semi 0.5.
    # POOL SIZE: the manifest holds 144,887 frames (verified). '--semi-n 300000' is a NON-BINDING CAP, not a 300k pool —
    # the "288k/300k" figures in the docs are double-counted. Coverage/epoch = 26 labeled batches x 10 steps x 256 = 66.5k.
    import os, shutil
    EXP = 'exp6'                                   # <-- bump per experiment; keeps Drive artifacts from clobbering
    if not os.path.exists('phase3/cache/unl_manifest.npz'):
        os.makedirs('phase3/cache', exist_ok=True)
        if os.path.exists(f'{DRIVE_DIR}/unl_manifest.npz'):
            shutil.copy(f'{DRIVE_DIR}/unl_manifest.npz', 'phase3/cache/unl_manifest.npz'); print('REUSED unl_manifest.npz')
        else:
            !python -m phase3.mine_hardneg --manifest-only
            shutil.copy('phase3/cache/unl_manifest.npz', f'{DRIVE_DIR}/unl_manifest.npz'); print('built unl_manifest.npz')
    FLAGS = (f'--backbone {BACKBONE} --img {FT_IMG} --init concept_encoder.pt '
             '--unfreeze 6 --wise-ft 0.7 --epochs 12 --aug domain '
             '--semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 '
             '--semi-n 300000 --semi-bs 256 --semi-steps 10 --ema-decay 0.99 '
             '--semi-use-decision --ship-ema')      # decision-gated PU target + keep the EMA teacher for the A/B
    for s in [0, 1, 2]:
        !python -m phase3.finetune --train-csv train_colab.csv --holdout none --seed {s} \
            {FLAGS} --bs 96 --loss bce+rank+pauc --warmup 2 --out ship_seed{s}.pt
    os.makedirs(f'{DRIVE_DIR}/{EXP}', exist_ok=True)
    for s in [0, 1, 2]:
        shutil.copy(f'ship_seed{s}.pt', f'{DRIVE_DIR}/{EXP}/ship_seed{s}.pt')      # versioned, never overwritten
        shutil.copy(f'ship_seed{s}.pt', f'{DRIVE_DIR}/ship_seed{s}.pt')            # "latest" convenience copy
        if os.path.exists(f'ship_seed{s}_ema.pt'):
            shutil.copy(f'ship_seed{s}_ema.pt', f'{DRIVE_DIR}/{EXP}/ship_seed{s}_ema.pt')
    print(f'DONE -> ship_seed0/1/2.pt ({EXP}: {BACKBONE} @{FT_IMG} + aug-domain + concept + semi + WiSE-FT, 12ep)')
    print(f'       archived to {DRIVE_DIR}/{EXP}/ . Watch the per-epoch "semi=" number: if it is orders of magnitude')
    print('       below the supervised loss, the unlabeled pool is decorative and --semi-mode fixmatch is the fix.')


SKIPPED: exp6 cell is superseded by MAX-A/B/C. Set RUN_LEGACY=True to run it.


In [ ]:
# ---- SUPERSEDED by MAX-A/B/C (cells 13-15). This full-power cell also writes ship_seed*.pt, so a
# ---- Run-All would clobber the MAX ensemble. Set RUN_LEGACY=True only to reproduce full-power.
RUN_LEGACY = False
if not RUN_LEGACY:
    print('SKIPPED: full-power cell is superseded by MAX-A/B/C. Set RUN_LEGACY=True to run it.')
else:
    # ==== FINAL SHIP — FULL POWER (the closed-phase container) ====================================
    # Uses EVERY source of signal we have. Run this ONCE, after the recipe is frozen by the gates below.
    #
    #   labeled   : train (2,476 / 127 pos) + val (619 source / 31 pos) = 3,095 frames, 158 pos  <- +24% POSITIVES
    #   concept   : Stage-1 on all 170,200 concept-annotated frames @FT_IMG (--holdout none = ship setting)
    #   semi      : all 144,887 VLM-scored frames, fixmatch pseudo-labels + decision-gated one-sided PU
    #   hard neg  : model-mined false positives from the VLM-negative pool -> --neg-list (the FP tail the metric reads)
    #   tail loss : --pauc-q at the REAL operating point (10th-pct positive) + OHEM margin + 16 pos/batch
    #   weights   : 3 seeds x (best-epoch + SWAD + EMA teacher), WiSE-FT 0.7 toward the concept init
    #
    # WHY THIS IS SAFE FOR THE SHIP BUT NOT FOR GATING: the hidden test was never in any of our data, so training on
    # val/semi/concept is legitimate and only helps. But once val is in training, val can no longer MEASURE anything.
    # => Freeze the recipe on the train-only gates FIRST. This cell is the last thing you run.
    import os, shutil, csv, json, glob
    EXP = 'final'

    # ---- 1. labeled = train + val (val JSONs are already de-augmented: 619 source frames, not the 8x copies) ----
    def build_rows(split):
        rows = []
        for j in glob.glob(f'out/{split}/labels/*.json'):
            d = json.load(open(j))
            if int(d.get('label', -1)) < 0: continue
            nm = d.get('name', os.path.splitext(os.path.basename(j))[0])
            img = f'out/{split}/images/{nm}.jpg'
            if os.path.exists(img):
                rows.append({'path': img, 'center': d.get('center', ''), 'class': '', 'label': int(d['label'])})
        return rows
    rows = build_rows('train') + build_rows('val')
    seen, dedup = set(), []
    for r in rows:                                        # hash-dedup in case a frame appears in both splits
        if r['path'] in seen: continue
        seen.add(r['path']); dedup.append(r)
    with open('trainval_colab.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['path', 'center', 'class', 'label']); w.writeheader(); w.writerows(dedup)
    NPOS = sum(r['label'] for r in dedup)
    print(f'trainval_colab.csv: {len(dedup)} frames, {NPOS} pos, {len(dedup)-NPOS} neg '
          f'(train-only was 2476 / 127 -> +{NPOS-127} positives, +{(NPOS-127)/127:.0%})')

    # ---- 2. mine model-FP hard negatives (the plumbing existed but no ship has ever used it) ----
    SCORERS = [p for p in ['ship_seed0.pt', 'ship_seed1.pt', 'ship_seed2.pt'] if os.path.exists(p)]
    NEG = ''
    if SCORERS:
        !python -m phase3.mine_hardneg --score-with {','.join(SCORERS)} \
            --pool CONFIDENT_NEGATIVE --topn 3000 --skip-top 200
        if os.path.exists('phase3/cache/unl_modelFP.txt'):
            NEG = '--neg-list phase3/cache/unl_modelFP.txt --neg-cap 3000'
            print('mined hard negatives ->', sum(1 for _ in open('phase3/cache/unl_modelFP.txt')), 'paths')
    else:
        print('NOTE: no ship_seed*.pt to score with -> skipping the hard-FP mine. Run the exp6 cell first, '
              'then re-run this cell to get the mined negatives (they add ~3k FP-tail negatives AND ~3x the '
              'optimizer steps per epoch, which is itself a large win at only ~26 batches/epoch today).')

    # ---- 3. the run ----
    FLAGS = (f'--backbone {BACKBONE} --img {FT_IMG} --init concept_encoder.pt '
             '--unfreeze 6 --wise-ft 0.7 --epochs 12 --aug domain '
             '--semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 '
             '--semi-n 300000 --semi-bs 256 --semi-steps 10 --ema-decay 0.99 '
             '--semi-mode fixmatch --semi-use-decision '        # pool contributes real gradient, PU gated on VLM decision
             '--pauc-q 0.0625 --pos-per-batch 16 --ohem-k 16 '  # defend the 10th-pct positive = where PPV@90R is read
             '--swad --swad-last-n 5 --ship-ema '               # flat-minima average + keep the teacher
             f'{NEG}')
    for s in [0, 1, 2]:
        !python -m phase3.finetune --train-csv trainval_colab.csv --holdout none --seed {s} \
            {FLAGS} --bs 96 --loss bce+rank+pauc --warmup 2 --out ship_seed{s}.pt

    os.makedirs(f'{DRIVE_DIR}/{EXP}', exist_ok=True)
    for s in [0, 1, 2]:
        for suf in ['', '_ema']:
            p = f'ship_seed{s}{suf}.pt'
            if os.path.exists(p): shutil.copy(p, f'{DRIVE_DIR}/{EXP}/{p}')
    print(f'\nDONE -> ship_seed0/1/2.pt archived to {DRIVE_DIR}/{EXP}/')
    print('Package cell next. Do NOT read val metrics after this — val is now training data.')


SKIPPED: full-power cell is superseded by MAX-A/B/C. Set RUN_LEGACY=True to run it.


In [ ]:
# ==== MAX-A: build the 100%-utilisation lists ==================================================
# Every one of the 288,711 unlabeled frames gets the role its LABEL RELIABILITY earns:
#   214,584 CONFIDENT_NEGATIVE  -> hard y=0            (VLM certain it is normal mucosa)
#      ~600 mined from HARD_NEG/ABSTAIN -> soft y=0.80 (4-gate: concept & bucket & suspicion & model)
#    73,525 rest of HARD_NEG/ABSTAIN    -> semi consistency ONLY, never y=0 (ARCHITECTURE.md section 5)
#
# WHY THE BIG NEGATIVE SET IS THE MAIN EVENT: PPV@90R = 0.9/(0.9+100*FPR@90R), and FPR@90R is literally
# the fraction of NEGATIVES above the threshold. exp6 learned "normal mucosa" from 2,349 negatives in
# 26 optimizer steps/epoch and memorised by ep7. This gives 217,521 negatives and 2,719 steps/epoch.
#
# WHY THE PSEUDO-POSITIVES ARE SMALL AND SOFT: measured on the 3,095 labeled frames, VLM suspicion>=0.9
# is 100% precise (n=32, zero FPs among 2,937 negatives). But the pool has 5.88% of frames at s>=0.9 vs
# 1.03% in the labeled set, so that precision may not transfer (>=17% if the pool is 1% positive, ~85%
# if 5%). Hence strict gates, a hard cap, and a soft target -- and MAX-B measures whether they help.
import os, shutil
os.makedirs('phase3/cache', exist_ok=True)

# 1. full manifest + the COMPLETE confident-negative list (default --confneg-sample is only 30k)
if not os.path.exists('phase3/cache/unl_confneg.txt') or sum(1 for _ in open('phase3/cache/unl_confneg.txt')) < 100000:
    !python -m phase3.mine_hardneg --confneg-sample 250000
N_NEG = sum(1 for _ in open('phase3/cache/unl_confneg.txt'))
print(f'confident-negative list: {N_NEG:,} frames')

# 2. pseudo-positives, strictest gates. Scored with the exp6 seeds (they never saw these pool frames).
# The MODEL gate needs a detector. On a fresh Colab nothing is trained yet, so pull the exp6 seeds from
# Drive (they were trained on train-only and have never seen a single pool frame, so scoring the pool with
# them is not leakage — only mild self-training bias, which the concept/suspicion gates are there to offset).
for _s in [0,1,2]:
    _d = f'{DRIVE_DIR}/exp6/ship_seed{_s}.pt'
    if not os.path.exists(f'ship_seed{_s}.pt') and os.path.exists(_d):
        shutil.copy(_d, f'ship_seed{_s}.pt'); print(f'pulled exp6 ship_seed{_s}.pt for the model gate')
SC = ','.join(p for p in ['ship_seed0.pt','ship_seed1.pt','ship_seed2.pt'] if os.path.exists(p))
if not SC:
    print('WARNING: no detector available -> the model gate is SKIPPED and pseudo-positive precision drops.')
    print('         Run exp6 (or one fold of MAX-B) first, then re-run this cell.')
!python -m phase3.mine_pseudopos --concept-targets phase3/cache/concept_targets.npz \
    --manifest phase3/cache/unl_manifest.npz \
    --susp-min 0.95 --pos-pct 60 --model-min 0.90 --topn 600 \
    {'--score-with ' + SC if SC else ''} \
    --out phase3/cache/unl_pseudopos.txt
N_POS = sum(1 for _ in open('phase3/cache/unl_pseudopos.txt')) if os.path.exists('phase3/cache/unl_pseudopos.txt') else 0
print(f'pseudo-positives: {N_POS} (on top of 158 labeled) -> +{N_POS/158:.0%}')

# 3. labeled = train + val (val JSONs are already de-augmented to 619 source frames)
import csv, json as _json, glob
def rows_for(split):
    out=[]
    for j in glob.glob(f'out/{split}/labels/*.json'):
        d=_json.load(open(j))
        if int(d.get('label',-1))<0: continue
        nm=d.get('name', os.path.splitext(os.path.basename(j))[0]); img=f'out/{split}/images/{nm}.jpg'
        if os.path.exists(img): out.append({'path':img,'center':d.get('center',''),'class':'','label':int(d['label'])})
    return out
r = rows_for('train') + rows_for('val')
seen=set(); r=[x for x in r if not (x['path'] in seen or seen.add(x['path']))]
with open('trainval_colab.csv','w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['path','center','class','label']); w.writeheader(); w.writerows(r)
print(f"trainval_colab.csv: {len(r)} frames, {sum(x['label'] for x in r)} positives")
for p in ['phase3/cache/unl_confneg.txt','phase3/cache/unl_pseudopos.txt']:
    if os.path.exists(p): shutil.copy(p, f'{DRIVE_DIR}/{os.path.basename(p)}')


scanned 288711 unlabeled frames
decision counts: {'CONFIDENT_NEGATIVE': 215986, 'HARD_NEG_CANDIDATE': 60323, 'ABSTAIN': 12402}
  unl_hardneg.txt: 60323 paths (of 60323 requested)
  unl_confneg.txt: 215986 paths (of 215986 requested)
  unl_suspicious.txt: 5000 paths (of 5000 requested)
suspicion: >0.7 -> 45885  >0.85 -> 25580
confident-negative list: 215,986 frames
pulled exp6 ship_seed0.pt for the model gate
pulled exp6 ship_seed1.pt for the model gate
pulled exp6 ship_seed2.pt for the model gate
concept matrix: 291,187 frames | decisive concepts used: ['demarcation', 'nodularity', 'vascular_irregularity', 'focal_abnormal_vessels', 'depression_ulceration', 'surface_effacement', 'dilated_vessels']
gate A (concept >= p60 of labeled positives = 0.6172): 40,347 / 288,711 unlabeled
gate C (decision in ['ABSTAIN', 'HARD_NEG_CANDIDATE']): 40,092 survive A+C
gate D (VLM suspicion >= 0.95): 2,381 survive A+C+D
scoring 2,381 candidates with ['ship_seed0.pt', 'ship_seed1.pt', 'ship_seed2.pt'] ...

In [ ]:
# ==== MAX-B: 5-fold CV — the ensemble AND the first honest measurement we have ==================
# Two things at once, which is why this is the core cell:
#   1. ENSEMBLE. IMSY won RARE25 with 40 models from 5-fold CV. We ship 3 seeds of one split. Fold
#      models are diverse by construction (each sees a different 80% of the labels).
#   2. MEASUREMENT. Pooled across folds, every one of the 158 labeled positives gets an OUT-OF-FOLD
#      prediction from a model that never saw it. That is the only uncontaminated validation signal
#      available: same-centre val is at ceiling (0.97 AUROC vs 0.86 on the board), and LOCO is
#      contaminated by the concept encoder. OOF AUPRC is the number to optimise — it is the metric
#      where we trail the winner 0.390 vs 0.822.
#
# 100% of the corpus is in play: 214,584 CONFIDENT_NEGATIVE as hard y=0, ~600 4-gate-mined
# pseudo-positives as soft y=0.80, and the ~74k ambiguous HARD_NEG/ABSTAIN frames as the semi pool
# (consistency belongs exactly where a hard label is unavailable — running it over frames we already
# label would be redundant, hence --semi-buckets).
import os, numpy as np
from phase3 import evaluate as ev

N_FOLDS, SEEDS = 5, [0]          # add seeds ([0,1,2] = 15 models) once the 5-fold OOF looks right
ARMS = {'pos':   '--pos-list phase3/cache/unl_pseudopos.txt --pos-soft 0.80',
        'nopos': ''}             # A/B the one component whose precision we could not derive

MAXCFG = (f'--backbone {BACKBONE} --img {FT_IMG} --init concept_encoder.pt --unfreeze 6 --wise-ft 0.7 '
          '--epochs 3 --bs 96 --pos-per-batch 12 --loss bce+rank+pauc --warmup 1 --aug domain '
          '--pauc-q 0.0625 --ohem-k 12 --ship-ema '
          '--neg-list phase3/cache/unl_confneg.txt --neg-cap 250000 '
          '--semi-manifest phase3/cache/unl_manifest.npz --semi-buckets HARD_NEG_CANDIDATE,ABSTAIN '
          '--semi-weight 0.5 --semi-n 300000 --semi-bs 96 --semi-steps 1 --semi-use-decision')
# NB semi_bs/steps: the semi pool is now only the ~74k ambiguous frames, and there are 2,589 labeled
# steps/epoch. steps=1 x bs=96 already sweeps it 3.4x per epoch; steps=4 x bs=192 would sweep it 27x
# and cost 110 GPU-hours instead of 19 for zero extra coverage.

# Colab disconnects. Every finished fold is mirrored to Drive and restored on a fresh session, so a
# ~20h sweep survives interruption and resumes exactly where it stopped.
import shutil, glob
FOLD_DIR = f'{DRIVE_DIR}/max_folds'; os.makedirs(FOLD_DIR, exist_ok=True)
for f in glob.glob(f'{FOLD_DIR}/*'):
    if not os.path.exists(os.path.basename(f)): shutil.copy(f, os.path.basename(f))
print(f'restored {len(glob.glob(f"{FOLD_DIR}/*.pt"))} checkpoint(s) from Drive')

for arm, extra in ARMS.items():
    for s in SEEDS:
        for k in range(N_FOLDS):
            out = f'fold{k}_{arm}_s{s}.pt'
            if os.path.exists(out): print(f'skip {out} (done)'); continue
            print(f'=== {arm} seed{s} fold {k}/{N_FOLDS} ===')
            !python -m phase3.finetune --train-csv trainval_colab.csv --fold {k} --n-folds {N_FOLDS} \
                --seed {s} {MAXCFG} {extra} --out {out}
            for suf in ['.pt', '_loco.npz', '_loco_final.npz', '_ema.pt']:
                src = out[:-3] + suf
                if os.path.exists(src): shutil.copy(src, f'{FOLD_DIR}/{os.path.basename(src)}')

# ---- pool the out-of-fold predictions and score them like the leaderboard does ----
def oof(arm, s=0, kind='final'):
    """kind='final' = the LAST-epoch scores (no selection -> UNBIASED, this is the honest number).
       kind='best'  = the AUPRC-selected epoch, which was chosen ON this very fold -> optimistic."""
    suf = '_loco_final.npz' if kind == 'final' else '_loco.npz'
    Y, C, S = [], [], []
    for k in range(N_FOLDS):
        f = f'fold{k}_{arm}_s{s}{suf}'
        if not os.path.exists(f): return None
        d = np.load(f, allow_pickle=True); Y.append(d['y']); C.append(d['c']); S.append(d['s'])
    return np.concatenate(Y), np.concatenate(C), np.concatenate(S)

print('\n=== OUT-OF-FOLD (every prediction from a model that never saw that frame) ===')
print('    "final" = last epoch, no selection = the HONEST estimate.')
print('    "best"  = AUPRC-selected epoch, selected on this same fold = optimistic, comparison only.\n')
res = {}
for kind in ('final', 'best'):
    for arm in ARMS:
        o = oof(arm, kind=kind)
        if o is None: print(f'  {arm}/{kind}: incomplete'); continue
        y, c, s = o
        if kind == 'final': res[arm] = (y, c, s)
        r = ev.report_full(y, s, c, B=2000)
        print(f"  {arm:6s}/{kind:5s} n={len(y)} pos={int(y.sum())} | AUROC={r['auroc']:.4f} "
              f"AUPRC={r['auprc']:.4f} | FPR@90R={r['fpr90']:.4f} "
              f"-> implied LB PPV {0.9/(0.9+100*r['fpr90']):.4f}")
print("\n  (exp6 scored LB AUROC 0.8602 / AUPRC 0.3900 / FPR@90R 0.453. OOF is same-centre so it will read")
print("   HIGHER than the board — track the DELTA between arms and across designs, not the absolute value.)")
if len(res) == 2:
    y, c, sp = res['pos']; _, _, sn = res['nopos']
    for m in ('auprc', 'auroc'):
        g = ev.gate(y, sp, sn, center=c, metric=m, B=4000)
        print(f"  pseudo-positive effect on {m}: {g['delta']:+.4f} CI[{g['lo']:+.4f},{g['hi']:+.4f}] {g['verdict']}")
    print('\nDECISION: set USE_PSEUDO in MAX-C from the AUPRC delta above.')


restored 0 checkpoint(s) from Drive
=== pos seed0 fold 0/5 ===
device=cuda | backbone=dinov2 | img=336 | cg_head=False
FOLD 0/5: train 2474 (125 pos) | held-out 621 (33 pos)
+ 215986 unlabeled negatives
+ 600 PSEUDO-positives @ soft target 0.8 (real positives: 125) — UNVERIFIED, gate before shipping
[init] loaded concept-pretrained backbone from concept_encoder.pt
train frames=219060 pos=725(real 125) | unfreeze last 6 blocks | trainable params=42.543M | holdout=center_2
sampler: 12 pos + 84 neg / batch, 2599 batches, pos_weight=7.0
/content/rare/phase3/finetune.py:604: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp = dev == "cuda"; scaler = torch.cuda.amp.GradScaler(enabled=amp)
⚠ LEAK WARNING: LOCO with the semi pool ON — unlabeled frames have no center label, so the held-out center may be in the pool (UDA to the test center) -> LOCO is OPTIMISTIC. Use --loco-no-semi for an honest compass, or trust 

In [ ]:
# ==== MAX-C: SHIP — ensemble the fold models (IMSY's actual method) =============================
# The 5 (or 15) fold models ARE the ensemble: each saw a different 80% of the labels, so they are
# diverse by construction, and every one of them was validated out-of-fold in MAX-B. No retrain on
# 100% of the labels is needed -- IMSY shipped exactly this.
USE_PSEUDO = True          # <-- set from the MAX-B AUPRC delta
ARM  = 'pos' if USE_PSEUDO else 'nopos'
SEEDS_SHIP, N_FOLDS = [0], 5
EXP = f'exp7_{ARM}'      # exp6 = the 0.0195 board run; this is exp7
import os, shutil, glob
members = [f'fold{k}_{ARM}_s{s}.pt' for s in SEEDS_SHIP for k in range(N_FOLDS) if os.path.exists(f'fold{k}_{ARM}_s{s}.pt')]
assert members, 'no fold models found — run MAX-B first'
print(f'ensembling {len(members)} members: {members}')
os.makedirs(f'{DRIVE_DIR}/{EXP}', exist_ok=True)
for i, m in enumerate(members):                       # the container globs resources/ship_seed*.pt
    shutil.copy(m, f'ship_seed{i}.pt'); shutil.copy(m, f'{DRIVE_DIR}/{EXP}/ship_seed{i}.pt')
    e = m[:-3] + '_ema.pt'
    if os.path.exists(e): shutil.copy(e, f'{DRIVE_DIR}/{EXP}/ship_seed{i}_ema.pt')
print(f'DONE -> {len(members)} members staged as ship_seed0..{len(members)-1}.pt, archived to {DRIVE_DIR}/{EXP}/')
print('Container: bash RARE25-Submission/build_submission.sh . exp7   (it verifies @336 mean-pool and runs the offline test)')
print('The OOF numbers from MAX-B are your honest estimate — do NOT re-score val, it is training data.')


ensembling 5 members: ['fold0_pos_s0.pt', 'fold1_pos_s0.pt', 'fold2_pos_s0.pt', 'fold3_pos_s0.pt', 'fold4_pos_s0.pt']
DONE -> 5 members staged as ship_seed0..4.pt, archived to /content/drive/MyDrive/RARE_LG/exp7_pos/
Container: bash RARE25-Submission/build_submission.sh . exp7   (it verifies @336 mean-pool and runs the offline test)
The OOF numbers from MAX-B are your honest estimate — do NOT re-score val, it is training data.


In [ ]:
# ==== HONEST GATE: does --aug domain beat mild on the NEW-center proxy @FT_IMG? ====
# TWO leaks must be closed for this to mean anything, and BOTH are now closed:
#   1. semi pool has no center labels -> --loco-no-semi drops it (was already here)
#   2. Stage-1 saw the held-out center's LABELED frames with label-proxy concepts -> use the
#      --holdout labeled gating encoder (cell 8, RUN_GATES=True). Without it this gate is optimistic.
# CAVEAT that remains: --loco-no-semi removes the pool, so this gate measures only the LABELED-aug
#   half of --aug domain, not the semi color-consistency half. Read it as a lower bound.
# Uses --loco-no-semi so the held-out center is NOT leaked via the (center-less) 288k semi pool -> honest compass.
# Trains each aug on each held-out center, reads the saved _loco.npz (y,center,scores), paired-bootstrap gate on AUROC.
# Ships NOTHING. Set RUN_AUG_GATE=True (~4 short finetunes). If aug-domain holds/improves on BOTH legs -> ship exp6.
RUN_AUG_GATE = False
if RUN_AUG_GATE:
    import os, numpy as np, phase3.evaluate as ev
    BASE = ('--backbone dinov2 --img {FT_IMG} --init concept_encoder_gate.pt --unfreeze 6 --wise-ft 0.7 '
            '--epochs 12 --bs 96 --loss bce+rank+pauc --warmup 2 --loco-no-semi '            # leak-free: no semi pool
            '--semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 --semi-n 300000 --semi-bs 256 --semi-steps 10')
    for hold in ['center_2', 'center_1']:
        for name, extra in [('mild', '--aug mild'), ('domain', '--aug domain')]:
            if not os.path.exists(f'augloco_{name}_{hold}_loco.npz'):
                print(f'--- holdout {hold} : {name} (leak-free) ---')
                !python -m phase3.finetune --train-csv train_colab.csv --seed 0 --holdout {hold} {BASE} {extra} --out augloco_{name}_{hold}.pt
    def leg(h, n):
        d = np.load(f'augloco_{n}_{h}_loco.npz', allow_pickle=True); return d['y'], d['c'], d['s']
    wins = 0
    for hold in ['center_2', 'center_1']:
        y, c, sd = leg(hold, 'domain'); _, _, sm = leg(hold, 'mild')
        print(f'\nholdout {hold} (aug-domain vs mild, leak-free):')
        for m in ('auroc', 'auprc'):
            g = ev.gate(y, sd, sm, center=c, metric=m, B=2000)
            print(f"  {m}: Δ={g['delta']:+.4f} CI[{g['lo']:+.4f},{g['hi']:+.4f}] -> {g['verdict']}")
            if m == 'auroc' and g['delta'] > 0: wins += 1
    print('\nDECISION:', 'aug-domain improves AUROC on BOTH new-center legs -> ship exp6 (cell above)' if wins == 2
          else 'aug-domain neutral/mixed -> weigh the risk; it is still the winner-proven color lever, ship remains defensible')
else:
    print('Aug-gate SKIPPED. Recommend RUN_AUG_GATE=True once: leak-free proof that --aug domain @336 helps the unseen '
          'center BEFORE spending the submission. (Set False to just ship exp6 on the winner-proven color-aug rationale.)')

Aug-gate SKIPPED. Recommend RUN_AUG_GATE=True once: leak-free proof that --aug domain @336 helps the unseen center BEFORE spending the submission. (Set False to just ship exp6 on the winner-proven color-aug rationale.)


## Package the submission (offline container)

`ship_seed{0,1,2}.pt` are the final weights. To submit:
1. Copy them into the container: `RARE25-Submission/resources/ship_seed{0,1,2}.pt`.
2. The container (`model/viscera_model.py`) loads them and runs **5-view TTA (orig/hflip/vflip/rot90/rot270) + 3-seed prob-mean** offline (`--network none`, per-image) — **identical** to the val cell below.
3. Build → test → save: run `do_test_run.sh`, then `do_save.sh`. **Validate the SAVED tar** (`gunzip -c <tar> | docker load` and run that image) before uploading — that's the exact artifact the platform runs.

The val cell below is a **same-center smoke test** = a mirage (optimistic; it read 0.65 before the real 0.018). It only confirms inference works; the honest new-center number is the **leaderboard**.

## Val scoring → competition metric (PPV@90R @1% prevalence, bootstrap median + CI)
Scores `val_colab.csv` with the 3-seed ensemble and reports the leaderboard metric the same way the grader does (curve-point PPV@90R, 1% prevalence, bootstrap), plus AUROC/AUPRC. ⚠️ **This is SAME-CENTER (both centers were in training) → optimistic** (it read 0.65 vs the real new-center 0.018). Use it only as a smoke test that inference works; the honest new-center number comes from **RARE25-val / the leaderboard**, not here. Read the **CI**, not the point.

In [ ]:
# ---- score val with the 3-seed ensemble + 5-VIEW TTA (EXACTLY the shipped container), report the 5 metrics ----
import os, csv, shutil, numpy as np
# restore ensemble from Drive if the runtime was reset
for s in [0, 1, 2]:
    if not os.path.exists(f'ship_seed{s}.pt') and os.path.exists(f'{DRIVE_DIR}/ship_seed{s}.pt'):
        shutil.copy(f'{DRIVE_DIR}/ship_seed{s}.pt', f'ship_seed{s}.pt')
assert all(os.path.exists(f'ship_seed{s}.pt') for s in [0, 1, 2]), 'ship_seed*.pt missing — run the ship cell or copy from Drive'
assert os.path.exists('val_colab.csv'), 'val_colab.csv missing — run the CSV-builder cell (needs out/val)'

# --tta 5view = orig/hflip/vflip/rot90/rot270 + prob-mean = IDENTICAL to RARE25-Submission/model/viscera_model.py,
# so this val number is what the offline container actually produces (not the old hflip-only approximation).
!python -m phase3.infer --model ship_seed0.pt,ship_seed1.pt,ship_seed2.pt --csv val_colab.csv --tta 5view --out preds.csv

from phase3 import evaluate as ev
# preds.csv = name,score,label ; pull center from val_colab.csv by frame name
center_by = {os.path.splitext(os.path.basename(r['path']))[0]: r.get('center', '')
             for r in csv.DictReader(open('val_colab.csv'))}
P = list(csv.DictReader(open('preds.csv')))
y = np.array([int(r['label']) for r in P]); s = np.array([float(r['score']) for r in P])
cen = np.array([center_by.get(r['name'], '') for r in P])
print(f'val frames={len(y)}  pos={int(y.sum())}  centers={sorted(set(cen))}  (TTA=5view = the container)\n')

hdr = f"{'split':10s} {'n':>4s} {'pos':>3s} | {'PPV@90R':>8s} {'CI_low':>7s} {'CI_high':>7s} | {'AUROC':>6s} {'AUPRC':>6s}"
print(hdr); print('-' * len(hdr))


def line(tag, yv, sv, cv):
    r = ev.report_full(yv, sv, cv if len(set(cv)) > 1 else None, target=0.9, prevalence=0.01, B=2000)
    print(f"{tag:10s} {r['n']:>4d} {r['pos']:>3d} | {r['ppv90']:>8.3f} {r['ci_lo']:>7.3f} {r['ci_hi']:>7.3f} | "
          f"{r['auroc']:>6.3f} {r['auprc']:>6.3f}")


line('POOLED', y, s, cen)
for cc in sorted(set(cen)):
    mk = cen == cc
    if mk.sum() and y[mk].sum() > 0:
        line(cc, y[mk], s[mk], cen[mk])
print("\nTTA=5view here == the offline container, so val predicts the leaderboard preprocessing faithfully.")
print("PPV@90R = curve-point @1% prevalence (leaderboard metric, HIGH variance at few pos — read the CI).")
print("AUROC/AUPRC = threshold-free ranking, STABLE at few pos, but NOT the 1%-operating-point score.")
print("SAME-CENTER val is optimistic (ship saw both centers); the honest new-center number is RARE25-val / the leaderboard.")

scoring 619 images (fine-tuned .pt)
  scored with ship_seed0.pt (tta=5view)
  scored with ship_seed1.pt (tta=5view)
  scored with ship_seed2.pt (tta=5view)
wrote 619 predictions -> preds.csv
val frames=619  pos=31  centers=[np.str_('center_1'), np.str_('center_2')]  (TTA=5view = the container)

split         n pos |  PPV@90R  CI_low CI_high |  AUROC  AUPRC
--------------------------------------------------------------
[bootstrap_challenge] WARNING: 588 negatives -> only 6 positive draws, so the recall>=0.9 point quantizes to an EFFECTIVE recall of 1.000. This number is not comparable to the leaderboard; use AUROC/AUPRC to rank recipes on a set this small.
POOLED      619  31 |    1.000   1.000   1.000 |  1.000  1.000
[bootstrap_challenge] WARNING: 444 negatives -> only 4 positive draws, so the recall>=0.9 point quantizes to an EFFECTIVE recall of 1.000. This number is not comparable to the leaderboard; use AUROC/AUPRC to rank recipes on a set this small.
center_1    456  12 |    1.000 

## D2F+ — decorrelated CNN member (LP-FT: Stage-1 concept→convergence, Stage-2 FROZEN encoder)
A **ConvNeXt-T / ResNet50** member makes DIFFERENT cross-center mistakes than the ViT anchor (local texture vs patch-token style) → rank-averaging cancels per-family center bias (the winner's ResNet50⊕ViT lever). Design (Kumar LP-FT, matches our frozen-LP=0.929 evidence): Stage-1 concept-teaching to convergence → Stage-2 **head-only (frozen encoder)** → preserves center-agnostic concept features. **Gate on LOCO before trusting it.**

In [ ]:
# D2F-1: CNN Stage-1 — concept-supervised to CONVERGENCE on the 144k pool. Cache to Drive.
import os, shutil
ARCH = 'convnext_tiny'   # LOCO: Tiny 0.932/0.976 > Large 0.909/0.965 -> Tiny wins (Large no gain, more compute). 'resnet50' alt.
CE = f'{DRIVE_DIR}/cnn_concept_{ARCH}.pt'
if os.path.exists(CE):
    shutil.copy(CE, 'cnn_concept.pt'); print('REUSED', CE)
else:
    !python -m phase3.cnn_member --stage concept --arch {ARCH} --targets phase3/cache/concept_targets.npz \
        --img 224 --epochs 30 --bs 128 --lr 1e-4 --out cnn_concept.pt   # 30ep = to convergence (watch loss plateau)
    shutil.copy('cnn_concept.pt', CE); print('saved', CE)

REUSED /content/drive/MyDrive/RARE_LG/cnn_concept_convnext_tiny.pt


In [ ]:
# D2F-2: LOCO GATE — does the CNN member DECORRELATE + help on the held-out center? (head-only, both legs)
# Trains head-only CNN with each center held out, scores the held-out center, and checks Spearman vs the ViT anchor.
RUN_CNN_GATE = True
if RUN_CNN_GATE:
    import numpy as np, csv, torch
    from phase3.cnn_member import CNNMember
    from sklearn.metrics import roc_auc_score
    from scipy.stats import spearmanr
    for hold in ['center_2','center_1']:
        !python -m phase3.cnn_member --stage finetune --arch {ARCH} --init cnn_concept.pt --unfreeze-stages 0 \
            --train-csv train_colab.csv --holdout {hold} --img 224 --epochs 20 --bs 96 --out cnn_loco_{hold}.pt
    rows=[r for r in csv.DictReader(open('val_colab.csv'))]; y=np.array([int(r['label']) for r in rows])
    cen=np.array([r['center'] for r in rows]); from PIL import Image; imgs=[Image.open(r['path']) for r in rows]
    for hold in ['center_2','center_1']:
        m=cen==hold; s=CNNMember(f'cnn_loco_{hold}.pt').score_frames([imgs[i] for i in np.where(m)[0]])
        print(f'  CNN LOCO holdout {hold}: AUROC={roc_auc_score(y[m],s):.3f}  (compare to ViT-B ~0.85-0.93)')
    print('DECISION: keep the CNN member only if its LOCO AUROC is decent (>~0.80) AND it decorrelates (Spearman<0.9 vs ViT).')
else: print('CNN gate skipped.')

[finetune] 1823 frames pos=49 holdout=center_2

model.safetensors: downloading bytes:  28% 31.6M/114M [00:00<00:01, 42.1MB/s,  497kB/s  ]
model.safetensors: downloading bytes:  44% 50.7M/114M [00:01<00:00, 66.9MB/s, 3.09MB/s  ]
model.safetensors: downloading bytes:  73% 83.7M/114M [00:01<00:00, 118MB/s, 4.92MB/s  ] 
model.safetensors: downloading bytes: 100% 109M/109M [00:01<00:00, 82.7MB/s, 10.4MB/s  ]
model.safetensors: reconstructing file: 100% 114M/114M [00:01<00:00, 87.0MB/s, 11.0MB/s  ]
[finetune] concept-init from cnn_concept.pt
[finetune] unfreeze_stages=0 -> trainable 0.0M (FROZEN encoder / head-only LP)
  ep1/20 loss=6.8646
  ep2/20 loss=4.7698
  ep3/20 loss=3.8114
  ep4/20 loss=3.2980
  ep5/20 loss=3.0403
  ep6/20 loss=2.8893
  ep7/20 loss=2.6917
  ep8/20 loss=2.6084
  ep9/20 loss=2.5400
  ep10/20 loss=2.4493
  ep11/20 loss=2.4560
  ep12/20 loss=2.4102
  ep13/20 loss=2.3719
  ep14/20 loss=2.2248
  ep15/20 loss=2.2530
  ep16/20 loss=2.2281
  ep17/20 loss=2.1758
  ep18/20 loss

In [ ]:
# D2F-3: CNN SHIP (head-only, both centers) + D2F+ ENSEMBLE val score (ViT anchor ⊕ CNN, rank-average)
import os, numpy as np, csv, glob, torch
from PIL import Image
from phase3.cnn_member import CNNMember
if not os.path.exists('cnn_member.pt'):
    !python -m phase3.cnn_member --stage finetune --arch {ARCH} --init cnn_concept.pt --unfreeze-stages 0 \
        --train-csv train_colab.csv --holdout none --img 224 --epochs 20 --bs 96 --out cnn_member.pt
    shutil.copy('cnn_member.pt', f'{DRIVE_DIR}/cnn_member_{ARCH}.pt')
rows=[r for r in csv.DictReader(open('val_colab.csv'))]; y=np.array([int(r['label']) for r in rows])
cen=np.array([r['center'] for r in rows]); imgs=[Image.open(r['path']) for r in rows]
def rank01(x): o=np.argsort(x,kind='mergesort');r=np.empty_like(o,float);r[o]=np.arange(len(x));return r/max(len(x)-1,1)
def fpr90(yy,ss,R=.9):
    P=yy.sum();N=(yy==0).sum();o=np.argsort(-ss);ys=yy[o];tp=np.cumsum(ys);fp=np.cumsum(1-ys);rc=tp/P
    return fp[min(np.searchsorted(rc,R),len(rc)-1)]/N
ppv1=lambda f:.01/(.01+.99*f)
# ViT anchor scores from infer.py's preds.csv (name,score,label) — align to `rows` BY FRAME NAME (never trust row order).
vit = None
if os.path.exists('preds.csv'):
    _sc = {r['name']: float(r['score']) for r in csv.DictReader(open('preds.csv'))}
    _names = [os.path.splitext(os.path.basename(r['path']))[0] for r in rows]
    if all(n in _sc for n in _names):
        vit = np.array([_sc[n] for n in _names])
    else:
        print(f"[warn] preds.csv missing {sum(n not in _sc for n in _names)}/{len(_names)} val frames — skipping ViT anchor (re-run cell 14)")
cnn = CNNMember('cnn_member.pt').score_frames(imgs)
members={'CNN':cnn}
if vit is not None and len(vit)==len(y): members['ViT-anchor']=vit
# rank-average (protects the tail vs one miscalibrated member)
ens = np.mean([rank01(s) for s in members.values()],0)
print('members:', list(members.keys()))
from sklearn.metrics import roc_auc_score
for tag,mask in [('POOLED',np.ones(len(y),bool)),('center_2',cen=='center_2')]:
    for nm,s in {**members,'D2F+ ens':ens}.items():
        f=fpr90(y[mask],s[mask]); print(f'  {tag:8} {nm:12} PPV@90R={ppv1(f):.3f} AUROC={roc_auc_score(y[mask],s[mask]):.3f}')
print('D2F+ wins only if the ensemble beats EACH member on center_2 (the honest leg). Gate on LOCO, not this same-center val.')

[finetune] 2476 frames pos=127 holdout=none
[finetune] concept-init from cnn_concept.pt
[finetune] unfreeze_stages=0 -> trainable 0.0M (FROZEN encoder / head-only LP)
  ep1/20 loss=6.1017
  ep2/20 loss=3.8916
  ep3/20 loss=3.0276
  ep4/20 loss=2.6018
  ep5/20 loss=2.3559
  ep6/20 loss=2.1809
  ep7/20 loss=2.0160
  ep8/20 loss=1.9536
  ep9/20 loss=1.8761
  ep10/20 loss=1.8203
  ep11/20 loss=1.8117
  ep12/20 loss=1.7134
  ep13/20 loss=1.6545
  ep14/20 loss=1.6454
  ep15/20 loss=1.6367
  ep16/20 loss=1.5972
  ep17/20 loss=1.5896
  ep18/20 loss=1.5793
  ep19/20 loss=1.5452
  ep20/20 loss=1.5078
[finetune] saved member -> cnn_member.pt
members: ['CNN', 'ViT-anchor']
  POOLED   CNN          PPV@90R=0.034 AUROC=0.915
  POOLED   ViT-anchor   PPV@90R=1.000 AUROC=1.000
  POOLED   D2F+ ens     PPV@90R=0.135 AUROC=0.976
  center_2 CNN          PPV@90R=0.016 AUROC=0.867
  center_2 ViT-anchor   PPV@90R=1.000 AUROC=1.000
  center_2 D2F+ ens     PPV@90R=0.038 AUROC=0.954
D2F+ wins only if the ensemble

## D2F-4 — HONEST LOCO harness: de-floor lever (A) + weighted ensemble gate (B)
The val cell (18) is **same-center = a mirage** and cannot judge either the de-floor lever or the ensemble. Cell **4a** builds the only valid bench — a pooled 2-center test where **every frame is scored by a model that never saw its center** (ViT-anchor LOCO @448 ⊕ ConvNeXt-CNN LOCO, both scoring the *same* held-out frames). Cell **4b** then runs the deep analysis:
- **(A) de-floor** (`SCORE_ALIGN_Q=0.10`): per-center score-shift cancellation. Key subtlety — de-floor is a **no-op within a single center** (constant shift = monotone); it only moves PPV@90R **across** the pooled centers. A positive Δ here = the lever really cancels the per-center shift.
- **(B) weighted ensemble**: sweep `w_ViT` in the rank-fuse `w·ViT+(1−w)·CNN`. equal-weight (w=0.5) was shown to **drag the tail** on val; this finds whether ANY weight (+de-floor) beats the ViT anchor on the honest proxy. If not → ship the anchor alone.

⚠ ~30 pos per leg → bootstrap CI is wide; read the CI, gate on direction + both legs, not the point estimate. (`infer.py` reads each ckpt's `img` from cfg, so @448 LOCO scores correctly.)

In [ ]:
# D2F-4a: HONEST LOCO proxy — every frame scored by a model BLIND to its center (both members, SAME frames).
# The val cell (18) is same-center = a MIRAGE and cannot judge de-floor or the ensemble. This can.
# Builds a pooled 2-center test where each frame is scored by a member that never saw its center.
import os, csv, numpy as np
from PIL import Image
from phase3.cnn_member import CNNMember
# ViT anchor LOCO = simple recipe (mean-pool + concept + semi, NO cg-head/mixstyle/aug-domain) at @448 (= the ship res).
ANCHOR = ('--backbone dinov2 --img 448 --init concept_encoder.pt '
          '--unfreeze 6 --wise-ft 0.7 --epochs 12 --bs 96 --loss bce+rank+pauc --warmup 2 '
          '--semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 --semi-n 300000 --semi-bs 192 --semi-steps 10')
allrows = list(csv.DictReader(open('val_colab.csv')))
legs = {}
for hold in ['center_2', 'center_1']:
    if not os.path.exists(f'vit_loco_{hold}.pt'):
        print(f'--- train ViT-anchor LOCO holdout {hold} (@448 simple recipe) ---')
        !python -m phase3.finetune --train-csv train_colab.csv --seed 0 --holdout {hold} {ANCHOR} --out vit_loco_{hold}.pt
    if not os.path.exists(f'cnn_loco_{hold}.pt'):
        print(f'--- train CNN LOCO holdout {hold} ({ARCH}) ---')
        !python -m phase3.cnn_member --stage finetune --arch {ARCH} --init cnn_concept.pt --unfreeze-stages 0 \
            --train-csv train_colab.csv --holdout {hold} --img 224 --epochs 20 --bs 96 --out cnn_loco_{hold}.pt
    sub = [r for r in allrows if r['center'] == hold]          # the honest test frames for this leg
    with open(f'_loco_val_{hold}.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=allrows[0].keys()); w.writeheader(); w.writerows(sub)
    !python -m phase3.infer --model vit_loco_{hold}.pt --csv _loco_val_{hold}.csv --tta 5view --out _vit_{hold}.csv
    vit_by = {r['name']: float(r['score']) for r in csv.DictReader(open(f'_vit_{hold}.csv'))}
    cnn = CNNMember(f'cnn_loco_{hold}.pt').score_frames([Image.open(r['path']) for r in sub])
    names = [os.path.splitext(os.path.basename(r['path']))[0] for r in sub]
    legs[hold] = dict(y=np.array([int(r['label']) for r in sub]),
                      vit=np.array([vit_by[n] for n in names]), cnn=np.array(cnn),
                      c=np.array([hold] * len(sub)))
    print(f'  {hold}: n={len(sub)} pos={int(legs[hold]["y"].sum())}')
Y = np.concatenate([legs[h]['y'] for h in legs]); C = np.concatenate([legs[h]['c'] for h in legs])
VIT = np.concatenate([legs[h]['vit'] for h in legs]); CNN = np.concatenate([legs[h]['cnn'] for h in legs])
np.savez('loco_proxy.npz', Y=Y, C=C, VIT=VIT, CNN=CNN)
print(f'POOLED honest proxy: n={len(Y)} pos={int(Y.sum())} centers={sorted(set(C))} -> loco_proxy.npz (run 4b)')

--- train ViT-anchor LOCO holdout center_2 (@448 simple recipe) ---
device=cuda | backbone=dinov2 | img=448 | cg_head=False
⚠ RESOLUTION MISMATCH: concept encoder concept_encoder.pt was pretrained @336 but Stage-2 runs @448. pos_embed will be interpolated, but the blocks were adapted to a different token grid — this is an UNCONTROLLED second variable. Rebuild Stage-1 with --img 448.
[init] loaded concept-pretrained backbone from concept_encoder.pt
train frames=1823 pos=49(real 49) | unfreeze last 6 blocks | trainable params=42.543M | holdout=center_2
sampler: 8 pos + 88 neg / batch, 20 batches, pos_weight=11.0
/content/rare/phase3/finetune.py:604: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp = dev == "cuda"; scaler = torch.cuda.amp.GradScaler(enabled=amp)
⚠ LEAK WARNING: LOCO with the semi pool ON — unlabeled frames have no center label, so the held-out center may be in the pool (UDA to the test cen

In [ ]:
# D2F-4b: DEEP ANALYSIS on the honest LOCO proxy — (A) de-floor lever, (B) weighted ensemble gate.
import numpy as np
from sklearn.metrics import roc_auc_score
d = np.load('loco_proxy.npz', allow_pickle=True); Y, C, VIT, CNN = d['Y'], d['C'], d['VIT'], d['CNN']
def rank01(x): o=np.argsort(x,kind='mergesort');r=np.empty_like(o,float);r[o]=np.arange(len(x));return r/max(len(x)-1,1)
def fpr90(y,s,R=.9):
    P=y.sum();N=(y==0).sum();o=np.argsort(-s);ys=y[o];tp=np.cumsum(ys);fp=np.cumsum(1-ys);rc=tp/max(P,1)
    return fp[min(np.searchsorted(rc,R),len(rc)-1)]/max(N,1)
ppv=lambda y,s: .01/(.01+.99*fpr90(y,s))
def defloor(s,c,q=.10):
    o=s.copy()
    for cv in np.unique(c): m=c==cv; o[m]=s[m]-np.quantile(s[m],q)
    return o
def boot(y,s,B=2000):
    rng=np.random.default_rng(0); idx=np.arange(len(y)); v=[]
    for _ in range(B):
        b=rng.choice(idx,len(idx),replace=True)
        if y[b].sum()>0: v.append(ppv(y[b],s[b]))
    return (np.median(v),np.quantile(v,.025),np.quantile(v,.975)) if v else (float('nan'),)*3

print(f'honest proxy: n={len(Y)} pos={int(Y.sum())}  (every frame scored by a center-BLIND model)\n')
print('=== (A) DE-FLOOR (SCORE_ALIGN_Q=0.10) — pooled honest proxy ===')
for tag,s in [('ViT raw',VIT),('ViT +defloor',defloor(VIT,C)),('CNN raw',CNN),('CNN +defloor',defloor(CNN,C))]:
    m,lo,hi=boot(Y,s); print(f'  {tag:14} PPV@90R={ppv(Y,s):.4f} boot_med={m:.4f}[{lo:.4f},{hi:.4f}] AUROC={roc_auc_score(Y,s):.3f}')
print('  NOTE: within ONE center, de-floor = constant shift = monotone = NO change. It only acts ACROSS pooled centers.')
print('  So a positive delta here = the lever cancels the per-center score-shift; ~0 = the two centers already align.')

print('\n=== (B) WEIGHTED ENSEMBLE — rank-fuse w*ViT+(1-w)*CNN, pooled honest proxy ===')
anchor=ppv(Y,VIT); anchor_d=ppv(Y,defloor(VIT,C)); best=(max(anchor,anchor_d),1.0,'anchor')
for w in [1.0,0.8,0.7,0.6,0.5,0.3,0.0]:
    ens=w*rank01(VIT)+(1-w)*rank01(CNN)
    p=ppv(Y,ens); pd=ppv(Y,defloor(ens,C))
    print(f'  w_ViT={w:.2f}  PPV@90R={p:.4f}  (+defloor={pd:.4f})  AUROC={roc_auc_score(Y,ens):.3f}')
    for val,lbl in [(p,f'w={w}'),(pd,f'w={w}+defloor')]:
        if val>best[0]: best=(val,w,lbl)
print(f'\n  ViT anchor alone: PPV@90R={anchor:.4f} (+defloor={anchor_d:.4f})   [bootstrap CI is WIDE at ~{int(Y.sum())} pos — read it]')
print(f'  BEST config     : PPV@90R={best[0]:.4f}  ({best[2]})')
print('  VERDICT:', 'ensemble/de-floor HELPS on the honest proxy -> ship it' if best[2]!='anchor'
      else 'NO gain over the ViT anchor -> ship the anchor alone (drop the CNN member)')

honest proxy: n=619 pos=31  (every frame scored by a center-BLIND model)

=== (A) DE-FLOOR (SCORE_ALIGN_Q=0.10) — pooled honest proxy ===
  ViT raw        PPV@90R=0.6644 boot_med=0.5431[0.1153,1.0000] AUROC=0.993
  ViT +defloor   PPV@90R=0.5429 boot_med=0.4949[0.1014,1.0000] AUROC=0.993
  CNN raw        PPV@90R=0.0613 boot_med=0.0616[0.0273,0.0930] AUROC=0.942
  CNN +defloor   PPV@90R=0.0177 boot_med=0.0178[0.0143,0.0338] AUROC=0.784
  NOTE: within ONE center, de-floor = constant shift = monotone = NO change. It only acts ACROSS pooled centers.
  So a positive delta here = the lever cancels the per-center score-shift; ~0 = the two centers already align.

=== (B) WEIGHTED ENSEMBLE — rank-fuse w*ViT+(1-w)*CNN, pooled honest proxy ===
  w_ViT=1.00  PPV@90R=0.6644  (+defloor=0.0241)  AUROC=0.993
  w_ViT=0.80  PPV@90R=0.3311  (+defloor=0.0234)  AUROC=0.993
  w_ViT=0.70  PPV@90R=0.2979  (+defloor=0.0228)  AUROC=0.990
  w_ViT=0.60  PPV@90R=0.2707  (+defloor=0.0209)  AUROC=0.986
  w_ViT=0.50  